### Structured output

Models can be requested to provide their response in a fomat matching a given schema . this is useful for ensuring the output can be parsed and used in subsequent processing . Langchain supports multiple schema types and method for enforcing structured output.

### Pydantic 

Pydantic models provide the richest feature set with field validation, descriptions and nested structures.

In [54]:
import os 
from langchain.chat_models import  init_chat_model
os.environ["GROQ_API_KEY"] = os.getenv("GROQ_API_KEY")
model = init_chat_model("openai/gpt-oss-120b", model_provider="groq")
model

ChatGroq(metadata={'lc_versions': {'langchain-core': '1.6.11', 'langchain': '1.4.6'}}, profile={'name': 'GPT OSS 120B', 'release_date': '2025-08-05', 'last_updated': '2026-05-27', 'open_weights': True, 'max_input_tokens': 131072, 'max_output_tokens': 65536, 'text_inputs': True, 'image_inputs': False, 'audio_inputs': False, 'video_inputs': False, 'text_outputs': True, 'image_outputs': False, 'audio_outputs': False, 'video_outputs': False, 'reasoning_output': True, 'tool_calling': True, 'structured_output': True, 'attachment': False, 'temperature': True}, client=<groq.resources.chat.completions.Completions object at 0x0000027FC6A75DF0>, async_client=<groq.resources.chat.completions.AsyncCompletions object at 0x0000027FC6A7D040>, model_name='openai/gpt-oss-120b', model_kwargs={}, groq_api_key=SecretStr('**********'))

In [55]:
from pydantic import BaseModel,Field

class Movie(BaseModel):
    title:str=Field(description="The title of the movie") # These all is Schema
    year:int=Field(description="This year the movie was released")
    director:str=Field(description="The director of the movie")
    rating:float=Field(description="The movie rating out of 10")

In [56]:
model_with_structure=model.with_structured_output(Movie)
model_with_structure

_ChatModelBinding(bound=ChatGroq(metadata={'lc_versions': {'langchain-core': '1.6.11', 'langchain': '1.4.6'}}, profile={'name': 'GPT OSS 120B', 'release_date': '2025-08-05', 'last_updated': '2026-05-27', 'open_weights': True, 'max_input_tokens': 131072, 'max_output_tokens': 65536, 'text_inputs': True, 'image_inputs': False, 'audio_inputs': False, 'video_inputs': False, 'text_outputs': True, 'image_outputs': False, 'audio_outputs': False, 'video_outputs': False, 'reasoning_output': True, 'tool_calling': True, 'structured_output': True, 'attachment': False, 'temperature': True}, client=<groq.resources.chat.completions.Completions object at 0x0000027FC6A75DF0>, async_client=<groq.resources.chat.completions.AsyncCompletions object at 0x0000027FC6A7D040>, model_name='openai/gpt-oss-120b', model_kwargs={}, groq_api_key=SecretStr('**********')), kwargs={'tools': [{'type': 'function', 'function': {'name': 'Movie', 'description': '', 'parameters': {'properties': {'title': {'description': 'The t

In [57]:
model.invoke("Provide details about the movie Mission Impossible 3")

AIMessage(content='**Mission:\u202fImpossible – Fallout (MI3)**  \n\n| Item | Details |\n|------|---------|\n| **Full Title** | *Mission:\u202fImpossible – Fallout* (commonly referred to as *Mission:\u202fImpossible\u202f3*) |\n| **Release Date** | May\u202f15\u202f2026 (United States) |\n| **Running Time** | 2\u202fhours\u202f14\u202fminutes |\n| **Genre** | Action‑Adventure, Spy Thriller |\n| **Production Companies** | Paramount Pictures, Skydance Media, Bad Robot Productions |\n| **Director** | Christopher McQuarrie (his third consecutive *Mission:\u202fImpossible* entry) |\n| **Screenplay** | Christopher\u202fMcQuarrie (based on the television series created by Bruce Geller) |\n| **Producers** | Tom\u202fCruise, Christopher\u202fMcQuarrie, David\u202fLevy, Jennifer\u202fL.\u202fMiller |\n| **Executive Producers** | J.J.\u202fAbrams, Brian\u202fKelley, Mark\u202fG.\u202fBaker |\n| **Principal Cast** | • **Tom Cruise** – Ethan Hunt (IMF field operative) <br>• **Rebecca Ferguson** – I

In [58]:
response= model_with_structure.invoke("Provide details about the movie Mission Impossible 3")
response

Movie(title='Mission: Impossible III', year=2006, director='J.J. Abrams', rating=7.1)

### Message Output alongside parsed structure

In [59]:
from pydantic import BaseModel,Field

class Movie(BaseModel):
    """ A movie with details."""
    title:str =Field(... ,description="The title of the movie") # These all is Schema
    year:int =Field(... ,description="This year the movie was released")
    director:str =Field(... ,description="The director of the movie")
    rating:float =Field(... ,description="The movie rating out of 10")

model_with_structure = model.with_structured_output(Movie, include_raw=True)

response = model_with_structure.invoke("Provide details about the movie Inception")
response

{'raw': AIMessage(content='', additional_kwargs={'reasoning_content': "User wants details about the movie Inception. We can use the function to get movie details. Provide director, rating, title, year. Let's call function.", 'tool_calls': [{'id': 'fc_071e1623-62a9-4895-8f19-2de006386c02', 'function': {'arguments': '{"director":"Christopher Nolan","rating":8.8,"title":"Inception","year":2010}', 'name': 'Movie'}, 'type': 'function'}]}, response_metadata={'token_usage': {'completion_tokens': 85, 'prompt_tokens': 163, 'total_tokens': 248, 'completion_time': 0.176517925, 'completion_tokens_details': {'reasoning_tokens': 33}, 'prompt_time': 0.006515778, 'prompt_tokens_details': None, 'queue_time': 0.24420028, 'total_time': 0.183033703}, 'model_name': 'openai/gpt-oss-120b', 'system_fingerprint': 'fp_ca5edfaab2', 'service_tier': 'on_demand', 'finish_reason': 'tool_calls', 'logprobs': None, 'model_provider': 'groq'}, id='lc_run--01a14e5b-4957-7800-b1c9-905cc8729f1b-0', tool_calls=[{'name': 'Mov

### Nested Structure

In [60]:
from pydantic import BaseModel, Field

class Actor(BaseModel):
    name:str
    role:str

class MovieDetails(BaseModel):
    title:str
    year:int
    cast:list[Actor]
    genre: list[str]
    budget: float | None = Field(None, description="Budget in million USD")

model_with_structure = model.with_structured_output(MovieDetails)

response = model_with_structure.invoke("Provide details about the movie Inception")
response

MovieDetails(title='Inception', year=2010, cast=[Actor(name='Leonardo DiCaprio', role='Dom Cobb'), Actor(name='Joseph Gordon-Levitt', role='Arthur'), Actor(name='Elliot Page', role='Ariadne'), Actor(name='Tom Hardy', role='Eames'), Actor(name='Ken Watanabe', role='Saito'), Actor(name='Cillian Murphy', role='Robert Fischer'), Actor(name='Marion Cotillard', role='Mal'), Actor(name='Michael Caine', role='Professor Stephen Miles')], genre=['Action', 'Adventure', 'Sci-Fi'], budget=160000000.0)

### TypedDict

TypeDict provides a simpler alternative using Python's built-in-typing ideal when you don't need runtime validation.

In [61]:
from typing_extensions import  TypedDict,Annotated

class MovieDict(TypedDict):
    """ A movie with details"""
    title: Annotated[str, ..., "The title of the movie"]
    year: Annotated[int, ..., "The year of the movie was released"]    
    director: Annotated[str, ..., "The director of the movie"]
    rating : Annotated[float, ..., "The movie's rating out of 10"]

model_withtypedict=model.with_structured_output(MovieDict)
response=model_withtypedict.invoke("Please details about of the movie Avenger")
response

{'director': 'Joss Whedon', 'rating': 8, 'title': 'The Avengers', 'year': 2012}

In [62]:
class Actor(TypedDict):
    name:str
    role:str

class MovieDetails(TypedDict):
    title:str
    year:int
    cast:list[Actor]
    genre: list[str]
    budget: float | None = Field(None, description="Budget in million USD")

model_with_structure = model.with_structured_output(MovieDetails)

response = model_with_structure.invoke("Provide details about the movie Avengers")
response

{'budget': 220000000,
 'cast': [{'name': 'Robert Downey Jr.', 'role': 'Tony Stark / Iron Man'},
  {'name': 'Chris Evans', 'role': 'Steve Rogers / Captain America'},
  {'name': 'Mark Ruffalo', 'role': 'Bruce Banner / Hulk'},
  {'name': 'Chris Hemsworth', 'role': 'Thor'},
  {'name': 'Scarlett Johansson', 'role': 'Natasha Romanoff / Black Widow'},
  {'name': 'Jeremy Renner', 'role': 'Clint Barton / Hawkeye'},
  {'name': 'Tom Hiddleston', 'role': 'Loki'},
  {'name': 'Samuel L. Jackson', 'role': 'Nick Fury'},
  {'name': 'Clark Gregg', 'role': 'Phil Coulson'},
  {'name': 'Cobie Smulders', 'role': 'Maria Hill'}],
 'genre': ['Action', 'Adventure', 'Sci-Fi', 'Superhero'],
 'title': 'The Avengers',
 'year': 2012}

In [63]:
model.profile

{'name': 'GPT OSS 120B',
 'release_date': '2025-08-05',
 'last_updated': '2026-05-27',
 'open_weights': True,
 'max_input_tokens': 131072,
 'max_output_tokens': 65536,
 'text_inputs': True,
 'image_inputs': False,
 'audio_inputs': False,
 'video_inputs': False,
 'text_outputs': True,
 'image_outputs': False,
 'audio_outputs': False,
 'video_outputs': False,
 'reasoning_output': True,
 'tool_calling': True,
 'structured_output': True,
 'attachment': False,
 'temperature': True}

### DataClasses

A data class is a class typically containing mainly data, althrough there aren't really any restrictions. You create it using the @dataclass decorator

In [78]:
import os
os.environ["GROQ_API_KEY"] = os.getenv("GROQ_API_KEY")

from pydantic import BaseModel, Field
from langchain.agents import  create_agent

class ContactInfo(BaseModel):
    """ Contact information for a person""" 
    name:str = Field(description="The name of the person")
    email:str = Field(description="The email address of the person")
    phone:str = Field(description="The phone number of the person")

agent = create_agent(
    model="groq:openai/gpt-oss-120b",
    response_format=ContactInfo   # Auto-select ProviderStrategy

)

result = agent.invoke({
    "messages" :[{"role":"user", "content":"Extract contact info from: John Doe ,john@example.com , (555),123-45676768"}]

})

result

{'messages': [HumanMessage(content='Extract contact info from: John Doe ,john@example.com , (555),123-45676768', additional_kwargs={}, response_metadata={}, id='00458ec5-0028-45a9-82a8-1b83062d5ec4'),
  AIMessage(content='{"name":"John Doe","email":"john@example.com","phone":"(555),123-45676768"}', additional_kwargs={'reasoning_content': 'The user asks: "Extract contact info from: John Doe ,john@example.com , (555),123-45676768". We need to output JSON according to ContactInfo schema: fields name, email, phone, all required.\n\nPhone number likely "(555),123-45676768". Probably combine as given? Should we format? Just output string.\n\nWe must ensure compact JSON, no whitespace besides necessary.\n\nThus output: {"name":"John Doe","email":"john@example.com","phone":"(555),123-45676768"}\n\nCheck schema: required fields present.\n\nReturn only JSON.'}, response_metadata={'token_usage': {'completion_tokens': 157, 'prompt_tokens': 238, 'total_tokens': 395, 'completion_time': 0.325291433, 

In [80]:
result["structured_response"]

ContactInfo(name='John Doe', email='john@example.com', phone='(555),123-45676768')

In [81]:
# TypeDict
from typing_extensions import  TypedDict
from langchain.agents import create_agent


class ContactInfo(TypedDict):
    """ Contact information for a person""" 
    name:str 
    email:str
    phone:str 

agent = create_agent(
    model="groq:openai/gpt-oss-120b",
    response_format=ContactInfo   # Auto-select ProviderStrategy

)

result = agent.invoke({
    "messages" :[{"role":"user", "content":"Extract contact info from: John Doe ,john@example.com , (555),123-45676768"}]

})

result["structured_response"]

{'name': 'John Doe', 'email': 'john@example.com', 'phone': '(555)123-45676768'}

In [83]:
# DataClass

from dataclasses import dataclass
from langchain.agents import create_agent

@dataclass
class ContactInfo:
    """ Contact information for a person""" 
    name:str # name of the person
    email:str # email address of the person
    phone:str # phone number of the person

agent = create_agent(
    model="groq:openai/gpt-oss-120b",
    response_format=ContactInfo
)

result = agent.invoke({
    "messages" :[{"role":"user", "content":"Extract contact info from: John Doe ,john@example.com , (555),123-45676768"}]
})

result["structured_response"]

ContactInfo(name='John Doe', email='john@example.com', phone='(555)123-45676768')